In [1]:
# Import librerie e setup percorsi
import glob
import os
import sys

from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# Import del modulo custom src/
sys.path.append(os.path.expanduser("~/tesi_graphrag"))
from src.graph_builder import KnowledgeGraphBuilder

# Percorsi e configurazioni -- DataSet 1 (ds1)
DATASET_DIR = os.path.expanduser("~/tesi_graphrag/data/raw/ds1")
COLLECTION_NAME = "ds1_graphrag_chunks"

print(f"Dataset directory: {DATASET_DIR}")

/tmp/ipykernel_6017/2182165333.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Dataset directory: /home/jovyan/tesi_graphrag/data/raw/ds1


In [2]:
# Caricamento PDF e Chunking

# Mappa dell'anno di pubblicazione ricavata dai paper
YEAR_MAP = {
    "GraphRAG_Microsoft": 2024,
    "Self_RAG": 2023,
    "CRAG_Corrective_RAG": 2024,
    "RAPTOR_Hierarchical_RAG": 2024,
    "Original_RAG_Lewis": 2020,
    "HyDE_Hypothetical_Embeddings": 2022,
    "KG_RAG_Survey": 2024,
    "FLARE_Active_RAG": 2023,
    "Knowledge_Graph_Prompting": 2023,
    "RAG_vs_FineTuning": 2024,
}

#! trova e elenca tutti i file contenuti nel percorso specificato
pdf_files = glob.glob(os.path.join(DATASET_DIR, "*.pdf"))
print(f"Trovati {len(pdf_files)} PDF in {DATASET_DIR}")

# Configurazione Text Splitter (512 token / 1000ca. caratteri, overlap di 150) 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, chunk_overlap = 150, length_function=len
)

all_docs = []

for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path).replace(".pdf", "")
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    # Split in chunk
    chunks = text_splitter.split_documents(pages)

    for idx, chunk in enumerate(chunks):
        page_num = chunk.metadata.get("page", 0)

        # Mantiene i metadati originali e aggiunge i nuovi campi di interesse
        chunk.metadata.update(
            {
                "doc_id": filename,
                "page": page_num,
                "year": YEAR_MAP.get(filename, 2024),
                "chunk_index": idx,
                "user_tags": [],  # Tag dinamici inizialmente vuoti per le manipolazioni successive
            }
        )

        all_docs.append(chunk)

print(f"Totale chunk generati: {len(all_docs)}")

Trovati 10 PDF in /home/jovyan/tesi_graphrag/data/raw/ds1
Totale chunk generati: 985


In [3]:
# Indicizzazione vettoriale su Qdrant
from src.config import (
    EMBEDDING_MODEL,
    QDRANT_URL,
)
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
import time

client = QdrantClient(url=QDRANT_URL)
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)

# Calcola automaticamente la dimensione corretta del modello attivo (es. 768)
vector_dim = len(embeddings.embed_query("test"))

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=client, collection_name=COLLECTION_NAME, embedding=embeddings
)

# Caricamento di tutti i chunk su Qdrant
#  Inserimento in batch di 25 chunk alla volta per vedere
#  se il processo si è bloccato o sta funzionando; + timer
batch_size = 25
total_docs = len(all_docs)
total_start_time = time.perf_counter()

print(
    f"\n>> Avvio vettorizzazione di {total_docs} chunk su CPU ({batch_size} chunk per batch)..."
)

# Ciclo di inserimento in batch con timer parziali e progressivo
for batch_idx, i in enumerate(range(0, total_docs, batch_size), start=1):
    batch_start_time = time.perf_counter()

    batch = all_docs[i : i + batch_size]
    vectorstore.add_documents(documents=batch)

    batch_elapsed = time.perf_counter() - batch_start_time
    processed_count = min(i + batch_size, total_docs)

    print(
        f"   [Batch {batch_idx:02d}] Indicizzati {processed_count:3d}/{total_docs} chunk "
        f"<T> tempo batch: {batch_elapsed:.2f} s"
    )

# Tempo totale
total_elapsed = time.perf_counter() - total_start_time

print("\n" + "=" * 55)
print(
    f"!!!> COMPLETATO! Inseriti {total_docs} chunk nella collezione '{COLLECTION_NAME}'"
)
print(f"<TT> TEMPO TOTALE VETTORIZZAZIONE & UPLOAD: {total_elapsed:.2f} s")
print("=" * 55)
print(f"Inseriti {len(all_docs)} chunk nella collezione Qdrant '{COLLECTION_NAME}'")


>> Avvio vettorizzazione di 985 chunk su CPU (25 chunk per batch)...
   [Batch 01] Indicizzati  25/985 chunk ⏱️ tempo batch: 19.03 s
   [Batch 02] Indicizzati  50/985 chunk ⏱️ tempo batch: 23.53 s
   [Batch 03] Indicizzati  75/985 chunk ⏱️ tempo batch: 17.38 s
   [Batch 04] Indicizzati 100/985 chunk ⏱️ tempo batch: 11.64 s
   [Batch 05] Indicizzati 125/985 chunk ⏱️ tempo batch: 11.76 s
   [Batch 06] Indicizzati 150/985 chunk ⏱️ tempo batch: 20.81 s
   [Batch 07] Indicizzati 175/985 chunk ⏱️ tempo batch: 14.54 s
   [Batch 08] Indicizzati 200/985 chunk ⏱️ tempo batch: 13.99 s
   [Batch 09] Indicizzati 225/985 chunk ⏱️ tempo batch: 14.73 s
   [Batch 10] Indicizzati 250/985 chunk ⏱️ tempo batch: 12.29 s
   [Batch 11] Indicizzati 275/985 chunk ⏱️ tempo batch: 10.27 s
   [Batch 12] Indicizzati 300/985 chunk ⏱️ tempo batch: 16.44 s
   [Batch 13] Indicizzati 325/985 chunk ⏱️ tempo batch: 15.19 s
   [Batch 14] Indicizzati 350/985 chunk ⏱️ tempo batch: 9.68 s
   [Batch 15] Indicizzati 375/985 c